We need the exact location of each port and to do that we have collated all the ports across the 4 files and put them in chatgpt to give us the accurate location stored it in ports.csv , and we have visualised them to make sure the port location is absolutely correct 

In [9]:
import pandas as pd
import re
from difflib import SequenceMatcher

# Read the port index
ports = pd.read_csv("data/World_Port_Index.csv")

# Read both vessel CSV files
cargill_vessels = pd.read_csv("data/Cargill_Capesize_Vessels.csv")
market_vessels = pd.read_csv("data/Market_Vessels_Formatted.csv")

print("Loaded files:")
print(f"Cargill vessels: {len(cargill_vessels)} rows")
print(f"Market vessels: {len(market_vessels)} rows")
print(f"Port index: {len(ports)} rows")

Loaded files:
Cargill vessels: 4 rows
Market vessels: 11 rows
Port index: 3669 rows


In [ ]:
import pandas as pd
import folium

# Load ports CSV
ports = pd.read_csv("ports.csv")

# Create base map (centered globally)
m = folium.Map(
    location=[20, 0],      # global view
    zoom_start=2,
    tiles="OpenStreetMap"
)

# Add ports as markers
for _, row in ports.iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=6,
        color="blue",
        fill=True,
        fill_color="blue",
        fill_opacity=0.7,
        tooltip=row["port_name"],  # hover tooltip
        popup=f"""
        <b>{row['port_name']}</b><br>
        Lat: {row['latitude']}<br>
        Lon: {row['longitude']}
        """
    ).add_to(m)

# Save map to HTML
m.save("ports_map.html")

print("Map saved as ports_map.html")


Map saved as ports_map.html


In [69]:
import re
import pandas as pd
from pathlib import Path

base = Path("data")

# 1. Read master port list (fixed format: port_name,latitude,longitude)
ports_df = pd.read_csv(base / "port_locations.csv")

# Build a lookup set from port_name
master_names = set()
for name in ports_df["port_name"].dropna():
    name = str(name).strip()
    if not name:
        continue
    master_names.add(name)
    master_names.add(name.lower())
    # Just the port part before comma, e.g. "Dampier" from "Dampier, Australia"
    short = name.split(",")[0].strip()
    master_names.add(short)
    master_names.add(short.lower())

def extract_port_only(val: str):
    """From things like 'Discharging Qingdao, China' or 'Qingdao, China' → 'Qingdao'."""
    if pd.isna(val):
        return None
    text = str(val).strip()

    # Strip common prefixes (for vessel status)
    text = re.sub(r"^(Discharging|Loading|At anchor|Anchored|At|In)\s+",
                  "", text, flags=re.IGNORECASE)

    # If 'Port, Country' keep just 'Port'
    m = re.match(r"^([^,]+),\s*(.+)$", text)
    if m:
        return m.group(1).strip()
    return text

def in_master(port: str) -> bool:
    if not port:
        return False
    variants = {port, port.strip(), port.lower().strip()}
    return any(v in master_names for v in variants)

# 2. Define files and the columns to check
files_and_cols = {
    "Cargill_Capesize_Vessels.csv": {
        "path": base.parent / "data" / "Cargill_Capesize_Vessels.csv",
        "cols": ["Current Position / Status"],
    },
    "Market_Vessels_Formatted.csv": {
        "path": base.parent / "data" / "Market_Vessels_Formatted.csv",
        "cols": ["Current Position / Status"],
    },
    "Cargill_Committed_Cargoes_Structured.csv": {
        "path": base.parent / "data" / "Cargill_Committed_Cargoes_Structured.csv",
        "cols": ["Load_Port_Primary", "Discharge_Port_Primary"],
    },
    "Market_Cargoes_Structured.csv": {
        "path": base.parent / "data" / "Market_Cargoes_Structured.csv",
        "cols": ["Load_Port", "Discharge_Port"],
    },
}

# 3. Check each file/column
for label, info in files_and_cols.items():
    df = pd.read_csv(info["path"])
    print(f"=== {label} ===")
    for col in info["cols"]:
        if col not in df.columns:
            print(f"  Column '{col}' not found in this file.")
            continue

        ports = (
            df[col]
            .dropna()
            .map(extract_port_only)
            .dropna()
            .unique()
        )

        missing = sorted(p for p in ports if not in_master(p))

        print(f"  Column: {col}")
        if not missing:
            print("    All ports exist in port_locations.csv")
        else:
            print("    Ports NOT found in port_locations.csv:")
            for p in missing:
                print(f"      - {p}")
    print()

=== Cargill_Capesize_Vessels.csv ===
  Column: Current Position / Status
    All ports exist in port_locations.csv

=== Market_Vessels_Formatted.csv ===
  Column: Current Position / Status
    All ports exist in port_locations.csv

=== Cargill_Committed_Cargoes_Structured.csv ===
  Column: Load_Port_Primary
    All ports exist in port_locations.csv
  Column: Discharge_Port_Primary
    All ports exist in port_locations.csv

=== Market_Cargoes_Structured.csv ===
  Column: Load_Port
    All ports exist in port_locations.csv
  Column: Discharge_Port
    All ports exist in port_locations.csv



In [72]:
import re
import pandas as pd
from pathlib import Path

base = Path("data")

# 1. Load master port locations: port_name like "Dampier_Australia"
ports_df = pd.read_csv(base / "port_locations.csv")

# Build mapping: "Dampier_Australia" -> (lat, lon)
port_lookup = {}
for _, row in ports_df.dropna(subset=["port_name"]).iterrows():
    key = str(row["port_name"]).strip().strip('"').strip()
    if key:
        port_lookup[key] = (row["latitude"], row["longitude"])

def make_port_key(val: str, is_status: bool = False):
    """
    Convert a value like:
      - 'Discharging Qingdao, China'
      - 'Qingdao, China'
    into a key like 'Qingdao_China' to look up in port_locations.csv.
    """
    if pd.isna(val):
        return None
    text = str(val).strip().strip('"').strip()

    if is_status:
        # Strip prefixes such as "Discharging", "Loading", "At", "In" etc.
        text = re.sub(
            r"^(Discharging|Loading|At anchor|Anchored|At|In)\s+",
            "",
            text,
            flags=re.IGNORECASE,
        ).strip()

    # Expect "Port, Country"
    m = re.match(r"^([^,]+),\s*(.+)$", text)
    if not m:
        # If no comma, we can't form "Port_Country" reliably
        return None

    port = m.group(1).strip()
    country = m.group(2).strip()
    key = f"{port}_{country}"
    return key

def lookup_lat_lon(val: str, is_status: bool = False):
    key = make_port_key(val, is_status=is_status)
    if not key:
        return pd.NA, pd.NA
    return port_lookup.get(key, (pd.NA, pd.NA))


# 2. Update Cargill_Capesize_Vessels.csv
print("Updating Cargill_Capesize_Vessels.csv ...")
cargill_vessels = pd.read_csv(base / "Cargill_Capesize_Vessels.csv")

lat_lon_series = cargill_vessels["Current Position / Status"].apply(
    lambda v: pd.Series(lookup_lat_lon(v, is_status=True), index=["Latitude", "Longitude"])
)
cargill_vessels[["Latitude", "Longitude"]] = lat_lon_series
cargill_vessels.to_csv(base / "Cargill_Capesize_Vessels.csv", index=False)
print("  ✓ Saved Cargill_Capesize_Vessels.csv")

# 3. Update Market_Vessels_Formatted.csv
print("Updating Market_Vessels_Formatted.csv ...")
market_vessels = pd.read_csv(base / "Market_Vessels_Formatted.csv")

lat_lon_series = market_vessels["Current Position / Status"].apply(
    lambda v: pd.Series(lookup_lat_lon(v, is_status=True), index=["Latitude", "Longitude"])
)
market_vessels[["Latitude", "Longitude"]] = lat_lon_series
market_vessels.to_csv(base / "Market_Vessels_Formatted.csv", index=False)
print("  ✓ Saved Market_Vessels_Formatted.csv")

# 4. Update Cargill_Committed_Cargoes_Structured.csv
print("Updating Cargill_Committed_Cargoes_Structured.csv ...")
cargill_cargoes = pd.read_csv(base / "Cargill_Committed_Cargoes_Structured.csv")

load_lat_lon = cargill_cargoes["Load_Port_Primary"].apply(
    lambda v: pd.Series(lookup_lat_lon(v, is_status=False), index=["Load_Port_Latitude", "Load_Port_Longitude"])
)
disc_lat_lon = cargill_cargoes["Discharge_Port_Primary"].apply(
    lambda v: pd.Series(lookup_lat_lon(v, is_status=False), index=["Discharge_Port_Latitude", "Discharge_Port_Longitude"])
)

for col in ["Load_Port_Latitude", "Load_Port_Longitude",
            "Discharge_Port_Latitude", "Discharge_Port_Longitude"]:
    if col not in cargill_cargoes.columns:
        cargill_cargoes[col] = pd.NA

cargill_cargoes[["Load_Port_Latitude", "Load_Port_Longitude"]] = load_lat_lon
cargill_cargoes[["Discharge_Port_Latitude", "Discharge_Port_Longitude"]] = disc_lat_lon

cargill_cargoes.to_csv(base / "Cargill_Committed_Cargoes_Structured.csv", index=False)
print("  ✓ Saved Cargill_Committed_Cargoes_Structured.csv")

# 5. Update Market_Cargoes_Structured.csv
print("Updating Market_Cargoes_Structured.csv ...")
market_cargoes = pd.read_csv(base / "Market_Cargoes_Structured.csv")

load_lat_lon = market_cargoes["Load_Port"].apply(
    lambda v: pd.Series(lookup_lat_lon(v, is_status=False), index=["Load_Port_Latitude", "Load_Port_Longitude"])
)
disc_lat_lon = market_cargoes["Discharge_Port"].apply(
    lambda v: pd.Series(lookup_lat_lon(v, is_status=False), index=["Discharge_Port_Latitude", "Discharge_Port_Longitude"])
)

for col in ["Load_Port_Latitude", "Load_Port_Longitude",
            "Discharge_Port_Latitude", "Discharge_Port_Longitude"]:
    if col not in market_cargoes.columns:
        market_cargoes[col] = pd.NA

market_cargoes[["Load_Port_Latitude", "Load_Port_Longitude"]] = load_lat_lon
market_cargoes[["Discharge_Port_Latitude", "Discharge_Port_Longitude"]] = disc_lat_lon

market_cargoes.to_csv(base / "Market_Cargoes_Structured.csv", index=False)
print("  ✓ Saved Market_Cargoes_Structured.csv")

print("\nAll four files updated from data/port_locations.csv.")

Updating Cargill_Capesize_Vessels.csv ...
  ✓ Saved Cargill_Capesize_Vessels.csv
Updating Market_Vessels_Formatted.csv ...
  ✓ Saved Market_Vessels_Formatted.csv
Updating Cargill_Committed_Cargoes_Structured.csv ...
  ✓ Saved Cargill_Committed_Cargoes_Structured.csv
Updating Market_Cargoes_Structured.csv ...
  ✓ Saved Market_Cargoes_Structured.csv

All four files updated from data/port_locations.csv.


In [73]:
import pandas as pd
import folium
from pathlib import Path

# Read Market cargoes file
base = Path("data")
market_cargoes = pd.read_csv(base / "Market_Cargoes_Structured.csv")

# Keep only rows with both load/discharge coordinates present
df = market_cargoes.dropna(
    subset=[
        "Load_Port_Latitude", "Load_Port_Longitude",
        "Discharge_Port_Latitude", "Discharge_Port_Longitude",
    ]
).copy()

if df.empty:
    print("No Market cargo rows with complete coordinates.")
else:
    # Compute map center
    center_lat = pd.concat([df["Load_Port_Latitude"], df["Discharge_Port_Latitude"]]).mean()
    center_lon = pd.concat([df["Load_Port_Longitude"], df["Discharge_Port_Longitude"]]).mean()

    # Create map
    m = folium.Map(location=[center_lat, center_lon], zoom_start=2)

    # Add each cargo as load/discharge markers and a route line
    for _, row in df.iterrows():
        route = row.get("Route", "")
        commodity = row.get("Commodity", "")
        qty = row.get("Quantity_MT", "")

        # Load marker (blue)
        folium.CircleMarker(
            location=[row["Load_Port_Latitude"], row["Load_Port_Longitude"]],
            radius=7,
            color="blue",
            fill=True,
            fill_color="blue",
            fill_opacity=0.8,
            tooltip=f"Load: {row['Load_Port']}",
            popup=(
                f"<b>Market Cargo – Load Port</b><br>"
                f"Route: {route}<br>"
                f"Port: {row['Load_Port']}<br>"
                f"Commodity: {commodity}<br>"
                f"Quantity: {qty} MT"
            ),
        ).add_to(m)

        # Discharge marker (red)
        folium.CircleMarker(
            location=[row["Discharge_Port_Latitude"], row["Discharge_Port_Longitude"]],
            radius=7,
            color="red",
            fill=True,
            fill_color="red",
            fill_opacity=0.8,
            tooltip=f"Discharge: {row['Discharge_Port']}",
            popup=(
                f"<b>Market Cargo – Discharge Port</b><br>"
                f"Route: {route}<br>"
                f"Port: {row['Discharge_Port']}<br>"
                f"Commodity: {commodity}<br>"
                f"Quantity: {qty} MT"
            ),
        ).add_to(m)

        # Line between load and discharge
        folium.PolyLine(
            locations=[
                [row["Load_Port_Latitude"], row["Load_Port_Longitude"]],
                [row["Discharge_Port_Latitude"], row["Discharge_Port_Longitude"]],
            ],
            color="purple",
            weight=2,
            opacity=0.6,
            tooltip=f"Route: {route}",
        ).add_to(m)

    # Simple legend
    legend_html = """
    <div style="
        position: fixed;
        bottom: 50px;
        left: 50px;
        width: 220px;
        background-color: white;
        border: 2px solid grey;
        z-index: 9999;
        font-size: 14px;
        padding: 10px;
    ">
      <b>Market Cargoes</b><br>
      <span style="color: blue;">●</span> Load Ports<br>
      <span style="color: red;">●</span> Discharge Ports<br>
      <span style="color: purple;">━</span> Cargo Route
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))

    # Save map
    out_path = Path("market_cargoes_map.html")
    m.save(str(out_path))
    print(f"✓ Saved Market cargoes map to '{out_path}'")

No Market cargo rows with complete coordinates.


In [79]:
import pandas as pd
from pathlib import Path

base = Path("data")

# 1. Load master port locations
ports_df = pd.read_csv(base / "port_locations.csv")

def norm(s):
    if pd.isna(s):
        return None
    return str(s).strip().strip('"').strip()

# Build lookup: normalized port_name -> (lat, lon)
port_lookup = {}
for _, row in ports_df.dropna(subset=["port_name"]).iterrows():
    key = norm(row["port_name"])
    if key:
        port_lookup[key] = (row["latitude"], row["longitude"])

# 2. Define the four CSVs and which columns contain the port names
files_and_cols = {
    "Cargill_Capesize_Vessels.csv": {
        "path": base / "Cargill_Capesize_Vessels.csv",
        "cols": ["Current Position / Status"],
    },
    "Market_Vessels_Formatted.csv": {
        "path": base / "Market_Vessels_Formatted.csv",
        "cols": ["Current Position / Status"],
    },
    "Cargill_Committed_Cargoes_Structured.csv": {
        "path": base / "Cargill_Committed_Cargoes_Structured.csv",
        "cols": ["Load_Port_Primary", "Discharge_Port_Primary"],
    },
    "Market_Cargoes_Structured.csv": {
        "path": base / "Market_Cargoes_Structured.csv",
        "cols": ["Load_Port", "Discharge_Port"],
    },
}

# 3. For each file/column: look up exact port_name in port_locations and write lat/lon beside it
for label, info in files_and_cols.items():
    path = info["path"]
    df = pd.read_csv(path)
    print(f"Updating {label} ...")

    for col in info["cols"]:
        if col not in df.columns:
            print(f"  - Column '{col}' not found, skipping")
            continue

        # Build lat/lon series by exact lookup
        lat_series = []
        lon_series = []
        for val in df[col]:
            key = norm(val)
            lat, lon = port_lookup.get(key, (pd.NA, pd.NA))
            lat_series.append(lat)
            lon_series.append(lon)

        lat_col = f"{col}_Latitude"
        lon_col = f"{col}_Longitude"

        # Insert directly after the port column
        idx = df.columns.get_loc(col)
        if lat_col in df.columns:
            df.drop(columns=[lat_col], inplace=True)
        if lon_col in df.columns:
            df.drop(columns=[lon_col], inplace=True)
        df.insert(idx + 1, lat_col, lat_series)
        df.insert(idx + 2, lon_col, lon_series)

        print(f"  - Wrote {lat_col}, {lon_col}")

    df.to_csv(path, index=False)
    print(f"  ✓ Saved {label}\n")

print("All four files updated from data/port_locations.csv.")

Updating Cargill_Capesize_Vessels.csv ...
  - Wrote Current Position / Status_Latitude, Current Position / Status_Longitude
  ✓ Saved Cargill_Capesize_Vessels.csv

Updating Market_Vessels_Formatted.csv ...
  - Wrote Current Position / Status_Latitude, Current Position / Status_Longitude
  ✓ Saved Market_Vessels_Formatted.csv

Updating Cargill_Committed_Cargoes_Structured.csv ...
  - Wrote Load_Port_Primary_Latitude, Load_Port_Primary_Longitude
  - Wrote Discharge_Port_Primary_Latitude, Discharge_Port_Primary_Longitude
  ✓ Saved Cargill_Committed_Cargoes_Structured.csv

Updating Market_Cargoes_Structured.csv ...
  - Wrote Load_Port_Latitude, Load_Port_Longitude
  - Wrote Discharge_Port_Latitude, Discharge_Port_Longitude
  ✓ Saved Market_Cargoes_Structured.csv

All four files updated from data/port_locations.csv.


In [83]:
import pandas as pd
from pathlib import Path

base = Path("data")

files = {
    "Cargill_Capesize_Vessels.csv": base / "Cargill_Capesize_Vessels.csv",
    "Market_Vessels_Formatted.csv": base / "Market_Vessels_Formatted.csv",
    "Cargill_Committed_Cargoes_Structured.csv": base / "Cargill_Committed_Cargoes_Structured.csv",
    "Market_Cargoes_Structured.csv": base / "Market_Cargoes_Structured.csv",
}

for label, path in files.items():
    print(f"\n=== {label} ===")
    df = pd.read_csv(path)
    print("Shape:", df.shape)
    print("\n.info():")
    df.info()
    print("\n.head():")
    display(df.head())


=== Cargill_Capesize_Vessels.csv ===
Shape: (4, 19)

.info():
<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 19 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   Vessel Name                                  4 non-null      str    
 1   DWT (MT)                                     4 non-null      int64  
 2   Hire Rate (USD/day)                          4 non-null      int64  
 3   Warranted Speed Laden (kn)                   4 non-null      float64
 4   Warranted Speed Ballast (kn)                 4 non-null      float64
 5   Warranted Sea Consumption Laden (mt/day)     4 non-null      int64  
 6   Warranted Sea Consumption Ballast (mt/day)   4 non-null      float64
 7   Economical Speed Laden (kn)                  4 non-null      float64
 8   Economical Speed Ballast (kn)                4 non-null      float64
 9   Economical Sea Consumption L

,Vessel Name,DWT (MT),Hire Rate (USD/day),Warranted Speed Laden (kn),Warranted Speed Ballast (kn),Warranted Sea Consumption Laden (mt/day),Warranted Sea Consumption Ballast (mt/day),Economical Speed Laden (kn),Economical Speed Ballast (kn),Economical Sea Consumption Laden (mt/day),Economical Sea Consumption Ballast (mt/day),Port Consumption Idle (mt/day),Port Consumption Working (mt/day),Current Position / Status,Current Position / Status_Latitude,Current Position / Status_Longitude,ETD,Bunker Remaining VLSFO (mt),Bunker Remaining MGO (mt)
0,ANN BELL,180803,11750,13.5,14.5,60,55.0,12.0,12.5,42,38.0,2.0,3.0,Qingdao_China,36.0669,120.3826,2026-02-25,401.3,45.1
1,OCEAN HORIZON,181550,15750,13.8,14.8,61,56.5,12.3,12.8,43,39.5,1.8,3.2,Map Ta Phut_Thailand,12.6512,101.1595,2026-03-01,265.8,64.3
2,PACIFIC GLORY,182320,14800,13.5,14.2,59,54.0,12.2,12.7,44,40.0,2.0,3.0,Gwangyang_South Korea,34.9074,127.6770,2026-03-10,601.9,98.1
3,GOLDEN ASCENT,179965,13950,13.0,14.0,58,53.0,11.8,12.3,41,37.0,1.9,3.1,Fangcheng_China,21.5878,108.3705,2026-03-08,793.3,17.1



=== Market_Vessels_Formatted.csv ===
Shape: (11, 18)

.info():
<class 'pandas.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 18 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   Vessel Name                                  11 non-null     str    
 1   DWT (MT)                                     11 non-null     int64  
 2   Warranted Speed Laden (kn)                   11 non-null     float64
 3   Warranted Speed Ballast (kn)                 11 non-null     float64
 4   Warranted Sea Consumption Laden (mt/day)     11 non-null     int64  
 5   Warranted Sea Consumption Ballast (mt/day)   11 non-null     float64
 6   Economical Speed Laden (kn)                  11 non-null     float64
 7   Economical Speed Ballast (kn)                11 non-null     float64
 8   Economical Sea Consumption Laden (mt/day)    11 non-null     float64
 9   Economical Sea Consumptio

,Vessel Name,DWT (MT),Warranted Speed Laden (kn),Warranted Speed Ballast (kn),Warranted Sea Consumption Laden (mt/day),Warranted Sea Consumption Ballast (mt/day),Economical Speed Laden (kn),Economical Speed Ballast (kn),Economical Sea Consumption Laden (mt/day),Economical Sea Consumption Ballast (mt/day),Port Consumption Idle (mt/day),Port Consumption Working (mt/day),Current Position / Status,Current Position / Status_Latitude,Current Position / Status_Longitude,ETD,Bunker Remaining VLSFO (mt),Bunker Remaining MGO (mt)
0,ATLANTIC FORTUNE,181200,13.8,14.6,60,56.0,12.3,12.9,43.0,39.5,2.0,3.0,Paradip_India,20.2777,86.6928,2026-03-02,512.4,38.9
1,PACIFIC VANGUARD,182050,13.6,14.3,59,54.0,12.0,12.5,42.0,38.0,1.9,3.0,Caofeidian_China,39.0150,118.4700,2026-02-26,420.3,51.0
2,CORAL EMPEROR,180450,13.4,14.1,58,53.0,11.9,12.3,40.0,36.5,2.0,3.1,Rotterdam_Netherlands,51.9400,4.1400,2026-03-05,601.7,42.3
3,EVEREST OCEAN,179950,13.7,14.5,61,56.5,12.4,12.8,43.5,39.0,1.8,3.0,Xiamen_China,24.5032,118.0833,2026-03-03,478.2,56.4
4,POLARIS SPIRIT,181600,13.9,14.7,62,57.0,12.5,13.0,44.0,40.0,2.0,3.1,Kandla_India,23.0300,70.2200,2026-02-28,529.8,47.1



=== Cargill_Committed_Cargoes_Structured.csv ===
Shape: (3, 24)

.info():
<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 24 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Route                        3 non-null      str    
 1   Customer                     3 non-null      str    
 2   Commodity                    3 non-null      str    
 3   Quantity_MT                  3 non-null      int64  
 4   Quantity_Tolerance           3 non-null      str    
 5   Laycan_Start                 3 non-null      str    
 6   Laycan_End                   3 non-null      str    
 7   Freight_Rate_USD_PMT         3 non-null      float64
 8   Load_Port_Primary            3 non-null      str    
 9   Load_Port_Latitude           3 non-null      float64
 10  Load_Port_Longitude          3 non-null      float64
 11  Load_Port_Notes              2 non-null      str    
 12  Load_Parcel_Size_M

,Route,Customer,Commodity,Quantity_MT,Quantity_Tolerance,Laycan_Start,Laycan_End,Freight_Rate_USD_PMT,Load_Port_Primary,Load_Port_Latitude,...,Discharge_Port_Primary,Discharge_Port_Latitude,Discharge_Port_Longitude,Discharge_Port_Alternatives,Discharge_Parcel_Size_MT,Discharge_Turn_Time_Hours,Port_Cost_USD,Port_Cost_Notes,Commission_Percent,Commission_Notes
0,West Africa – China,EGA,Bauxite,180000,+/- 10% Owners Option,2026-04-02,2026-04-10,23.0,Kamsar_Guinea,10.6392,...,Qingdao_China,36.0669,120.3826,Other Chinese ports allowed on same TCE basis,25000,12,0,"Nil, borne by charterer",1.25,Due to broker
1,Australia – China,BHP,Iron Ore,160000,"+/- 10%, half freight applies above 176,000 MT",2026-03-07,2026-03-11,9.0,Port Hedland_Australia,-20.3100,...,Lianyungang_China,34.7529,119.3770,NaN,30000,24,380000,USD 260K load + USD 120K discharge,3.75,Due to charterer
2,Brazil – China,CSN,Iron Ore,180000,+/- 10% MOLoo,2026-04-01,2026-04-08,22.3,Itaguaí_Brazil,-22.9332,...,Qingdao_China,36.0669,120.3826,Other Far East ports including river ports,30000,24,165000,USD 75K load + USD 90K discharge,3.75,Due to charterer



=== Market_Cargoes_Structured.csv ===
Shape: (8, 24)

.info():
<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 24 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Route                        8 non-null      str    
 1   Customer                     8 non-null      str    
 2   Commodity                    8 non-null      str    
 3   Quantity_MT                  8 non-null      int64  
 4   Quantity_Tolerance           8 non-null      str    
 5   Laycan_Start                 8 non-null      str    
 6   Laycan_End                   8 non-null      str    
 7   Load_Port                    8 non-null      str    
 8   Load_Port_Latitude           8 non-null      float64
 9   Load_Port_Longitude          8 non-null      float64
 10  Load_Turn_Size_MT            8 non-null      int64  
 11  Load_Turn_Time_hr            8 non-null      int64  
 12  Load_Terms_Notes             

,Route,Customer,Commodity,Quantity_MT,Quantity_Tolerance,Laycan_Start,Laycan_End,Load_Port,Load_Port_Latitude,Load_Port_Longitude,...,Discharge_Port_Latitude,Discharge_Port_Longitude,Alt_Discharge_Ports_Allowed,Discharge_Turn_Size_MT,Discharge_Turn_Time_hr,Discharge_Terms_Notes,Port_Cost_Load_USD,Port_Cost_Discharge_USD,Commission_Percent,Commission_Payable_To
0,Australia – China (Iron Ore),Rio Tinto,Iron Ore,170000,+/-10% MOLOO,2026-03-12,2026-03-18,Dampier_Australia,-20.6720,116.6990,...,36.0669,120.3826,No,30000,24,PWWD SHINC,240000,NaN,3.75,Charterer
1,Brazil – China (Iron Ore),Vale,Iron Ore,190000,+/-10% MOLOO,2026-04-03,2026-04-10,Ponta da Madeira_Brazil,-2.5370,-44.1870,...,39.0150,118.4700,No,30000,24,PWWD SHINC,75000,95000.0,3.75,Charterer
2,South Africa – China (Iron Ore),Anglo American,Iron Ore,180000,+/-10% MOLOO,2026-03-15,2026-03-22,Saldanha Bay_South Africa,-33.0111,17.9446,...,39.0036,117.7132,No,25000,24,PWWD SHINC,180000,NaN,3.75,Charterer
3,Indonesia – India (Coal),Adaro,Thermal Coal,150000,+/-10% MOLOO,2026-04-10,2026-04-15,Taboneo_Indonesia,-0.8033,117.1344,...,14.2470,80.1246,No,25000,24,PWWD SHINC,90000,NaN,2.50,Broker
4,Canada – China (Coking Coal),Teck Resources,Coking Coal,160000,+/-10% MOLOO,2026-03-18,2026-03-26,Vancouver_Canada,49.2965,-123.1338,...,21.5878,108.3705,No,25000,24,PWWD SHINC,180000,110000.0,3.75,Charterer


In [81]:
import pandas as pd
from pathlib import Path

base = Path("data")

# 1. Cargill_Capesize_Vessels: remove Latitude, Longitude
print("Cleaning Cargill_Capesize_Vessels.csv ...")
cargill_vessels_path = base / "Cargill_Capesize_Vessels.csv"
cv = pd.read_csv(cargill_vessels_path)
for col in ["Latitude", "Longitude"]:
    if col in cv.columns:
        cv = cv.drop(columns=[col])
cv.to_csv(cargill_vessels_path, index=False)
print("  ✓ Saved Cargill_Capesize_Vessels.csv")

# 2. Market_Vessels_Formatted: remove Latitude, Longitude
print("Cleaning Market_Vessels_Formatted.csv ...")
market_vessels_path = base / "Market_Vessels_Formatted.csv"
mv = pd.read_csv(market_vessels_path)
for col in ["Latitude", "Longitude"]:
    if col in mv.columns:
        mv = mv.drop(columns=[col])
mv.to_csv(market_vessels_path, index=False)
print("  ✓ Saved Market_Vessels_Formatted.csv")

# 3. Cargill_Committed_Cargoes_Structured:
#    remove Load_Port_Latitude, Load_Port_Longitude,
#           Discharge_Port_Latitude, Discharge_Port_Longitude
print("Cleaning Cargill_Committed_Cargoes_Structured.csv ...")
cargill_cargo_path = base / "Cargill_Committed_Cargoes_Structured.csv"
cc = pd.read_csv(cargill_cargo_path)
for col in [
    "Load_Port_Latitude",
    "Load_Port_Longitude",
    "Discharge_Port_Latitude",
    "Discharge_Port_Longitude",
]:
    if col in cc.columns:
        cc = cc.drop(columns=[col])
cc.to_csv(cargill_cargo_path, index=False)
print("  ✓ Saved Cargill_Committed_Cargoes_Structured.csv")

print("\nMarket_Cargoes_Structured.csv: nothing to remove; left unchanged.")

Cleaning Cargill_Capesize_Vessels.csv ...
  ✓ Saved Cargill_Capesize_Vessels.csv
Cleaning Market_Vessels_Formatted.csv ...
  ✓ Saved Market_Vessels_Formatted.csv
Cleaning Cargill_Committed_Cargoes_Structured.csv ...
  ✓ Saved Cargill_Committed_Cargoes_Structured.csv

Market_Cargoes_Structured.csv: nothing to remove; left unchanged.
